In [ ]:
# Load the mobility partitions and extract unique home and park coordinates.
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")

PART_FILES = sorted(OUTPUT_DIR.glob("01_user_park_big_table_part_*.csv"))
SINGLE_FILE = OUTPUT_DIR / "01_user_park_big_table.csv"

wanted_cols = [
    "user_ID", "osm_id", "park_name", "visit_count",
    "home_Lng", "home_Lat",
    "park_centroid_lng", "park_centroid_lat",
    "target_lng", "target_lat",
    "target_type", "target_source",
    "straight_distance_m"
]

if len(PART_FILES) > 0:
    print("Partitioned input files found; loading and concatenating them...")
    dfs = []
    for f in PART_FILES:
        tmp = pd.read_csv(f, usecols=lambda c: c in wanted_cols)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
elif SINGLE_FILE.exists():
    print("Single input file found; loading it...")
    df = pd.read_csv(SINGLE_FILE, usecols=lambda c: c in wanted_cols)
else:
    raise FileNotFoundError("Neither the combined input table nor its partitions were found.")

df = df.dropna(subset=["home_Lng", "home_Lat", "target_lng", "target_lat"]).copy()

print("Mobility table loaded")
print("Rows:", len(df))
display(df.head())


In [ ]:
# Load the home-node and park-node assignments.
home_points = pd.read_csv(OUTPUT_DIR / "home_points_with_nodes.csv")
target_points = pd.read_csv(OUTPUT_DIR / "target_points_with_nodes.csv")

print("home_points Rows:", len(home_points))
print("target_points Rows:", len(target_points))

display(home_points.head())
display(target_points.head())


In [ ]:
# Join network-node identifiers to mobility records using normalized spatial keys.
import pandas as pd
import numpy as np

keep_cols = [
    "user_ID", "osm_id", "park_name", "visit_count",
    "home_Lng", "home_Lat",
    "park_centroid_lng", "park_centroid_lat",
    "target_lng", "target_lat",
    "target_type", "target_source",
    "straight_distance_m"
]
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

df["user_ID"] = df["user_ID"].astype(str)
df["osm_id"] = df["osm_id"].astype(str)

home_points["user_ID"] = home_points["user_ID"].astype(str)
target_points["osm_id"] = target_points["osm_id"].astype(str)

df["target_lng_r"] = df["target_lng"].round(6)
df["target_lat_r"] = df["target_lat"].round(6)

target_points["target_lng_r"] = target_points["target_lng"].round(6)
target_points["target_lat_r"] = target_points["target_lat"].round(6)

df["target_key"] = (
    df["osm_id"] + "||" +
    df["target_lng_r"].astype(str) + "||" +
    df["target_lat_r"].astype(str)
)

target_points["target_key"] = (
    target_points["osm_id"] + "||" +
    target_points["target_lng_r"].astype(str) + "||" +
    target_points["target_lat_r"].astype(str)
)

home_map = (
    home_points.drop_duplicates(subset=["user_ID"])
    .set_index("user_ID")["home_node"]
)

target_map = (
    target_points.drop_duplicates(subset=["target_key"])
    .set_index("target_key")["target_node"]
)

df["home_node"] = df["user_ID"].map(home_map)
df["target_node"] = df["target_key"].map(target_map)

print("Node identifiers joined")
print("Rows:", len(df))
print("Missing home_node values:", df["home_node"].isna().sum())
print("Missing target_node values:", df["target_node"].isna().sum())

print("Duplicate osm_id values in target_points:", target_points["osm_id"].duplicated().sum())
print("Duplicate target_key values in target_points:", target_points["target_key"].duplicated().sum())

display(df[[
    "user_ID", "osm_id", "visit_count",
    "home_Lng", "home_Lat",
    "target_lng", "target_lat",
    "home_node", "target_node"
]].head())


In [ ]:
# Construct unique home-park node pairs within the distance cutoff.
from pathlib import Path
import numpy as np
import pandas as pd

CUTOFF_M = 10000
PAIR_FILE = OUTPUT_DIR / "node_pairs_for_network_distance.csv"

pair_base = df[[
    "home_node",
    "target_node",
    "straight_distance_m"
]].copy()

pair_df = (
    pair_base
    .groupby(["home_node", "target_node"], as_index=False)["straight_distance_m"]
    .min()
)

print("Unique node pairs:", len(pair_df))

# The straight-line cutoff restricts costly network routing to plausible walking-access pairs.
pair_df["pair_status"] = np.where(
    pair_df["straight_distance_m"] <= CUTOFF_M,
    "to_compute",
    "beyond_cutoff"
)

print(pair_df["pair_status"].value_counts())

pair_df.to_csv(PAIR_FILE, index=False, encoding="utf-8-sig")
print("Saved:", PAIR_FILE)

display(pair_df.head())


In [ ]:
# Calculate shortest-path distances in target-node batches.
import pandas as pd
import numpy as np
import networkx as nx
import osmnx as ox
from pathlib import Path

CUTOFF_M = 10000
TARGET_BATCH_SIZE = 20

OUTPUT_DIR = Path("outputs")
PAIR_FILE = OUTPUT_DIR / "node_pairs_for_network_distance.csv"
GRAPHML_FILE = OUTPUT_DIR / "walk_network_from_research_area.graphml"
BATCH_DIR = OUTPUT_DIR / "network_distance_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

if "Gp" not in globals():
    print("Gp is not in memory; rebuilding the projected network from GraphML...")
    G = ox.load_graphml(GRAPHML_FILE)
    G = ox.distance.add_edge_lengths(G)
    Gu = ox.convert.to_undirected(G)
    Gp = ox.project_graph(Gu)
    print("Gp rebuilt")

pair_df = pd.read_csv(PAIR_FILE)

pair_compute = pair_df[pair_df["pair_status"] == "to_compute"][["home_node", "target_node"]].copy()
pair_compute["home_node"] = pair_compute["home_node"].astype("int64")
pair_compute["target_node"] = pair_compute["target_node"].astype("int64")

print("Pairs requiring network routing:", len(pair_compute))

grouped = pair_compute.groupby("target_node")
target_nodes_list = sorted(grouped.groups.keys())

print("Unique target nodes to process:", len(target_nodes_list))

for batch_index, start in enumerate(range(0, len(target_nodes_list), TARGET_BATCH_SIZE), start=1):
    end = min(start + TARGET_BATCH_SIZE, len(target_nodes_list))
    batch_targets = target_nodes_list[start:end]

    batch_file = BATCH_DIR / f"network_distance_batch_{batch_index:04d}.csv"
    if batch_file.exists():
        print("Existing batch retained:", batch_file.name)
        continue

    print(f"Processing batch {batch_index}: target-node indices {start} to {end} ...")

    rows = []

    for target_node in batch_targets:
        grp = grouped.get_group(target_node)
        needed_home_nodes = grp["home_node"].tolist()

        # One search per destination reuses path lengths for all associated home nodes.
        lengths = nx.single_source_dijkstra_path_length(
            Gp,
            source=int(target_node),
            cutoff=CUTOFF_M,
            weight="length"
        )

        for home_node in needed_home_nodes:
            rows.append({
                "home_node": int(home_node),
                "target_node": int(target_node),
                "network_distance_m": lengths.get(int(home_node), np.nan)
            })

    batch_df = pd.DataFrame(rows)
    batch_df.to_csv(batch_file, index=False, encoding="utf-8-sig")
    print("Saved:", batch_file.name, "Rows:", len(batch_df))

print("All batches completed")


In [ ]:
# Require an empty batch directory before a new network-distance run.
from pathlib import Path

OUTPUT_DIR = Path("outputs")
BATCH_DIR = OUTPUT_DIR / "network_distance_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

existing_batches = sorted(BATCH_DIR.glob("network_distance_batch_*.csv"))
if existing_batches:
    raise RuntimeError(
        f"{BATCH_DIR} contains {len(existing_batches)} existing batch files. "
        "Move them to an archive directory before starting a new run."
    )


In [ ]:
# Load, project, and convert the walking network for shortest-path analysis.
import osmnx as ox

GRAPHML_FILE = OUTPUT_DIR / "walk_network_from_research_area.graphml"

print("Loading the cached walking network...")
G = ox.load_graphml(GRAPHML_FILE)
print("Original network:", G)

G = ox.distance.add_edge_lengths(G)

Gp = ox.project_graph(G)
print("Projected network:", Gp)

Gpu = ox.convert.to_undirected(Gp)
print("Undirected network used for shortest paths:", Gpu)


In [ ]:
# Calculate batched shortest paths on the projected undirected network.
import pandas as pd
import numpy as np
import networkx as nx
import osmnx as ox
from pathlib import Path

OUTPUT_DIR = Path("outputs")
PAIR_FILE = OUTPUT_DIR / "node_pairs_for_network_distance.csv"
GRAPHML_FILE = OUTPUT_DIR / "walk_network_from_research_area.graphml"
BATCH_DIR = OUTPUT_DIR / "network_distance_batches"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF_M = 15000
TARGET_BATCH_SIZE = 20

if "Gpu" not in globals():
    print("Gpu is not in memory; rebuilding it from GraphML...")
    G = ox.load_graphml(GRAPHML_FILE)
    G = ox.distance.add_edge_lengths(G)
    Gp = ox.project_graph(G)
    Gpu = ox.convert.to_undirected(Gp)
    print("Gpu rebuilt")

pair_df = pd.read_csv(PAIR_FILE)

pair_compute = pair_df[pair_df["pair_status"] == "to_compute"][["home_node", "target_node"]].copy()
pair_compute["home_node"] = pair_compute["home_node"].astype("int64")
pair_compute["target_node"] = pair_compute["target_node"].astype("int64")

grouped = pair_compute.groupby("target_node")
target_nodes_list = sorted(grouped.groups.keys())

print("Unique target nodes to process:", len(target_nodes_list))

for batch_index, start in enumerate(range(0, len(target_nodes_list), TARGET_BATCH_SIZE), start=1):
    end = min(start + TARGET_BATCH_SIZE, len(target_nodes_list))
    batch_targets = target_nodes_list[start:end]

    batch_file = BATCH_DIR / f"network_distance_batch_{batch_index:04d}.csv"
    if batch_file.exists():
        print("Existing batch retained:", batch_file.name)
        continue

    print(f"Processing batch {batch_index}: target-node indices {start} to {end} ...")

    rows = []

    for target_node in batch_targets:
        grp = grouped.get_group(target_node)
        needed_home_nodes = grp["home_node"].tolist()

        lengths = nx.single_source_dijkstra_path_length(
            Gpu,
            source=int(target_node),
            cutoff=CUTOFF_M,
            weight="length"
        )

        for home_node in needed_home_nodes:
            rows.append({
                "home_node": int(home_node),
                "target_node": int(target_node),
                "network_distance_m": lengths.get(int(home_node), np.nan)
            })

    batch_df = pd.DataFrame(rows)
    batch_df.to_csv(batch_file, index=False, encoding="utf-8-sig")
    print("Saved:", batch_file.name, "Rows:", len(batch_df))

print("All batches completed")


In [ ]:
# Aggregate batch results and attach route status to each node pair.
from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path("outputs")
PAIR_FILE = OUTPUT_DIR / "node_pairs_for_network_distance.csv"
BATCH_DIR = OUTPUT_DIR / "network_distance_batches"

batch_files = sorted(BATCH_DIR.glob("network_distance_batch_*.csv"))
print("Batch files found:", len(batch_files))

dist_df = pd.concat([pd.read_csv(f) for f in batch_files], ignore_index=True)

print("Rows after combining batch results:", len(dist_df))
print("Missing network_distance_m values:", dist_df["network_distance_m"].isna().sum())
print("Missing share of network_distance_m:", dist_df["network_distance_m"].isna().mean())

pair_df = pd.read_csv(PAIR_FILE)

pair_df = pair_df.merge(
    dist_df,
    on=["home_node", "target_node"],
    how="left"
)

pair_df["route_status"] = np.where(
    pair_df["pair_status"] == "beyond_cutoff",
    "beyond_cutoff",
    np.where(pair_df["network_distance_m"].notna(), "ok", "no_path")
)

print(pair_df["route_status"].value_counts(dropna=False))

pair_result_file = OUTPUT_DIR / "node_pairs_with_route_status.csv"
pair_df.to_csv(pair_result_file, index=False, encoding="utf-8-sig")

print("Written:", pair_result_file)
display(pair_df.head())


In [ ]:
# Merge route results with mobility records and export partitioned derived tables.
from pathlib import Path
import pandas as pd
import numpy as np
import math

OUTPUT_DIR = Path("outputs")

PAIR_RESULT_FILE = OUTPUT_DIR / "node_pairs_with_route_status.csv"

PART_FILES = sorted(OUTPUT_DIR.glob("01_user_park_big_table_part_*.csv"))
SINGLE_FILE = OUTPUT_DIR / "01_user_park_big_table.csv"

HOME_NODE_FILE = OUTPUT_DIR / "home_points_with_nodes.csv"
TARGET_NODE_FILE = OUTPUT_DIR / "target_points_with_nodes.csv"

FINAL_OUT = OUTPUT_DIR / "06_user_park_big_table_final_with_network_distance.csv"
MAX_ROWS_PER_FILE = 500000

wanted_cols = [
    "user_ID", "osm_id", "park_name", "visit_count",
    "home_Lng", "home_Lat",
    "park_centroid_lng", "park_centroid_lat",
    "target_lng", "target_lat",
    "target_type", "target_source",
    "straight_distance_m"
]

if len(PART_FILES) > 0:
    print("Partitioned input files found; loading and concatenating them...")
    dfs = []
    for f in PART_FILES:
        tmp = pd.read_csv(f, usecols=lambda c: c in wanted_cols)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
elif SINGLE_FILE.exists():
    print("Single input file found; loading it...")
    df = pd.read_csv(SINGLE_FILE, usecols=lambda c: c in wanted_cols)
else:
    raise FileNotFoundError("The original mobility table was not found.")

print("Rows in the original mobility table:", len(df))

home_points = pd.read_csv(HOME_NODE_FILE)
target_points = pd.read_csv(TARGET_NODE_FILE)

df["user_ID"] = df["user_ID"].astype(str)
df["osm_id"] = df["osm_id"].astype(str)

home_points["user_ID"] = home_points["user_ID"].astype(str)
target_points["osm_id"] = target_points["osm_id"].astype(str)

df["target_lng_r"] = df["target_lng"].round(6)
df["target_lat_r"] = df["target_lat"].round(6)

target_points["target_lng_r"] = target_points["target_lng"].round(6)
target_points["target_lat_r"] = target_points["target_lat"].round(6)

df["target_key"] = (
    df["osm_id"] + "||" +
    df["target_lng_r"].astype(str) + "||" +
    df["target_lat_r"].astype(str)
)

target_points["target_key"] = (
    target_points["osm_id"] + "||" +
    target_points["target_lng_r"].astype(str) + "||" +
    target_points["target_lat_r"].astype(str)
)

home_map = (
    home_points.drop_duplicates(subset=["user_ID"])
    .set_index("user_ID")["home_node"]
)

target_map = (
    target_points.drop_duplicates(subset=["target_key"])
    .set_index("target_key")["target_node"]
)

df["home_node"] = df["user_ID"].map(home_map)
df["target_node"] = df["target_key"].map(target_map)

print("Network nodes joined")
print("Missing home_node values:", df["home_node"].isna().sum())
print("Missing target_node values:", df["target_node"].isna().sum())

pair_result = pd.read_csv(PAIR_RESULT_FILE)

pair_result["pair_key"] = (
    pair_result["home_node"].astype("int64").astype(str) + "||" +
    pair_result["target_node"].astype("int64").astype(str)
)

df["pair_key"] = (
    df["home_node"].astype("Int64").astype(str) + "||" +
    df["target_node"].astype("Int64").astype(str)
)

network_map = pair_result.drop_duplicates(subset=["pair_key"]).set_index("pair_key")["network_distance_m"]
status_map = pair_result.drop_duplicates(subset=["pair_key"]).set_index("pair_key")["route_status"]

df["network_distance_m"] = df["pair_key"].map(network_map)
df["route_status"] = df["pair_key"].map(status_map)

print("Network distances and route status joined")
print(df["route_status"].value_counts(dropna=False))
print("Missing network_distance_m values:", df["network_distance_m"].isna().sum())

final_cols = [
    "user_ID", "osm_id", "park_name", "visit_count",
    "home_Lng", "home_Lat",
    "park_centroid_lng", "park_centroid_lat",
    "target_lng", "target_lat",
    "target_type", "target_source",
    "straight_distance_m",
    "network_distance_m",
    "route_status"
]

final_cols = [c for c in final_cols if c in df.columns]
df_out = df[final_cols].copy()

print("Rows in the final table:", len(df_out))
display(df_out.head())

def export_large_csv(df_out, output_base_path, max_rows=500000):
    n = len(df_out)
    if n <= max_rows:
        df_out.to_csv(output_base_path, index=False, encoding="utf-8-sig")
        print(f"Written: {output_base_path}")
        return

    num_parts = math.ceil(n / max_rows)
    stem = output_base_path.stem
    suffix = output_base_path.suffix

    for i in range(num_parts):
        start = i * max_rows
        end = min((i + 1) * max_rows, n)
        part_file = output_base_path.parent / f"{stem}_part_{i+1:03d}{suffix}"
        df_out.iloc[start:end].to_csv(part_file, index=False, encoding="utf-8-sig")
        print(f"Written: {part_file}  Rows: {end-start}")

export_large_csv(df_out, FINAL_OUT, max_rows=MAX_ROWS_PER_FILE)


In [ ]:
# Inspect identifier matching and route-result coverage before data derivation.
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE_0 = Path("data/restricted/mobility/mobility_part_0.csv")
RAW_FILE_1 = Path("data/restricted/mobility/mobility_part_1.csv")

OUTPUT_DIR = Path("outputs")
FINAL_PART_FILES = sorted(OUTPUT_DIR.glob("06_user_park_big_table_final_with_network_distance_part_*.csv"))

print("Final table partitions found:", len(FINAL_PART_FILES))
for f in FINAL_PART_FILES[:10]:
    print("  ", f.name)

def normalize_id_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        num = pd.to_numeric(s, errors="raise")
        if pd.notna(num) and np.isfinite(num) and abs(num - int(num)) < 1e-9:
            return str(int(num))
    except Exception:
        pass
    return s

def normalize_id_series(series):
    return series.map(normalize_id_value)

if len(FINAL_PART_FILES) == 0:
    raise FileNotFoundError("No final network-distance table partitions were found.")

want_cols = [
    "user_ID", "osm_id",
    "network_distance_m", "route_status",
    "straight_distance_m",
    "target_type", "target_source"
]

final_list = []
for f in FINAL_PART_FILES:
    tmp = pd.read_csv(f, usecols=lambda c: c in want_cols)
    final_list.append(tmp)

final_df = pd.concat(final_list, ignore_index=True)

print("\nFinal combined table loaded")
print("final_df Rows:", len(final_df))
print("Columns in final_df:", final_df.columns.tolist())

final_df["_merge_user_ID"] = normalize_id_series(final_df["user_ID"])
final_df["_merge_osm_id"] = normalize_id_series(final_df["osm_id"])

dup_final = final_df.duplicated(subset=["_merge_user_ID", "_merge_osm_id"]).sum()
print("Duplicate merge keys in final_df:", dup_final)

if dup_final > 0:
    print("Warning: duplicate keys exist in the final table; the first record is retained for diagnostics.")
    final_df = final_df.drop_duplicates(subset=["_merge_user_ID", "_merge_osm_id"], keep="first").copy()

route_lookup = final_df[[
    "_merge_user_ID", "_merge_osm_id",
    "network_distance_m", "route_status",
    "straight_distance_m", "target_type", "target_source"
]].copy()

route_lookup = route_lookup.rename(columns={
    "straight_distance_m": "straight_distance_m_final",
    "target_type": "target_type_final",
    "target_source": "target_source_final"
})

print("route_lookup Rows:", len(route_lookup))
display(route_lookup.head())

def inspect_merge(raw_path):
    print("\n" + "="*80)
    print("Checking file:", raw_path.name)

    raw_df = pd.read_csv(raw_path)
    print("Original rows:", len(raw_df))
    print("Original columns:", raw_df.columns.tolist())

    cols = list(raw_df.columns)

    user_col = "user_ID" if "user_ID" in raw_df.columns else cols[0]
    osm_col = "osm_id" if "osm_id" in raw_df.columns else cols[3]

    print("user_ID merge column:", user_col)
    print("osm_id merge column:", osm_col)

    tmp = raw_df.copy()
    tmp["_merge_user_ID"] = normalize_id_series(tmp[user_col])
    tmp["_merge_osm_id"] = normalize_id_series(tmp[osm_col])

    dup_raw = tmp.duplicated(subset=["_merge_user_ID", "_merge_osm_id"]).sum()
    print("Duplicate merge keys in the original table:", dup_raw)

    merged = tmp.merge(
        route_lookup,
        on=["_merge_user_ID", "_merge_osm_id"],
        how="left"
    )

    matched_any = merged["route_status"].notna().sum()
    unmatched = merged["route_status"].isna().sum()
    match_rate = matched_any / len(merged) if len(merged) > 0 else np.nan

    print("Rows after enrichment:", len(merged))
    print("Matched rows (non-null route_status):", matched_any)
    print("Unmatched rows:", unmatched)
    print("Match rate:", round(match_rate * 100, 2), "%")

    print("\nroute_status counts (including missing values):")
    print(merged["route_status"].value_counts(dropna=False))

    preview_cols = [user_col, osm_col]
    for c in ["steps", "park_class", "network_distance_m", "route_status", "straight_distance_m_final", "target_type_final"]:
        if c in merged.columns and c not in preview_cols:
            preview_cols.append(c)

    print("\nPreview:")
    display(merged[preview_cols].head(10))

    print("\nFirst 10 unmatched records:")
    display(
        merged.loc[merged["route_status"].isna(), [user_col, osm_col]].head(10)
    )

    return merged

merged_0_preview = inspect_merge(RAW_FILE_0)
merged_1_preview = inspect_merge(RAW_FILE_1)


In [ ]:
# Write mobility tables with network-distance fields to the derived-output directory.
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE_0 = Path("data/restricted/mobility/mobility_part_0.csv")
RAW_FILE_1 = Path("data/restricted/mobility/mobility_part_1.csv")

OUTPUT_DIR = Path("outputs")
DERIVED_DIR = OUTPUT_DIR / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PART_FILES = sorted(
    OUTPUT_DIR.glob("06_user_park_big_table_final_with_network_distance_part_*.csv")
)

if not FINAL_PART_FILES:
    raise FileNotFoundError(
        "No network-distance result parts were found in the outputs directory."
    )

def normalize_id_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    try:
        number = pd.to_numeric(text, errors="raise")
        if pd.notna(number) and np.isfinite(number) and abs(number - int(number)) < 1e-9:
            return str(int(number))
    except Exception:
        pass
    return text

def normalize_id_series(series):
    return series.map(normalize_id_value)

lookup_columns = [
    "user_ID",
    "osm_id",
    "network_distance_m",
    "route_status",
    "straight_distance_m",
    "target_type",
    "target_source",
]

parts = [
    pd.read_csv(path, usecols=lambda column: column in lookup_columns)
    for path in FINAL_PART_FILES
]
route_data = pd.concat(parts, ignore_index=True)
route_data["_merge_user_ID"] = normalize_id_series(route_data["user_ID"])
route_data["_merge_osm_id"] = normalize_id_series(route_data["osm_id"])
route_data = route_data.drop_duplicates(
    subset=["_merge_user_ID", "_merge_osm_id"], keep="first"
)

route_lookup = route_data[[
    "_merge_user_ID",
    "_merge_osm_id",
    "network_distance_m",
    "route_status",
    "straight_distance_m",
    "target_type",
    "target_source",
]].rename(columns={
    "straight_distance_m": "straight_distance_m_final",
    "target_type": "target_type_final",
    "target_source": "target_source_final",
})

def write_derived(source_path, output_path):
    source = pd.read_csv(source_path)
    original_columns = source.columns.tolist()
    user_column = "user_ID" if "user_ID" in source.columns else original_columns[0]
    park_column = "osm_id" if "osm_id" in source.columns else original_columns[3]

    prepared = source.copy()
    prepared["_merge_user_ID"] = normalize_id_series(prepared[user_column])
    prepared["_merge_osm_id"] = normalize_id_series(prepared[park_column])

    derived_columns = [
        "network_distance_m",
        "route_status",
        "straight_distance_m_final",
        "target_type_final",
        "target_source_final",
    ]
    prepared = prepared.drop(
        columns=[column for column in derived_columns if column in prepared.columns]
    )
    merged = prepared.merge(
        route_lookup,
        on=["_merge_user_ID", "_merge_osm_id"],
        how="left",
    ).drop(columns=["_merge_user_ID", "_merge_osm_id"])
    merged = merged[original_columns + derived_columns]
    merged.to_csv(output_path, index=False, encoding="utf-8-sig")
    return merged

DERIVED_FILE_0 = DERIVED_DIR / "mobility_part_0_with_network_distance.csv"
DERIVED_FILE_1 = DERIVED_DIR / "mobility_part_1_with_network_distance.csv"
merged_0_final = write_derived(RAW_FILE_0, DERIVED_FILE_0)
merged_1_final = write_derived(RAW_FILE_1, DERIVED_FILE_1)


In [ ]:
# Define shared estimators and settings for distance-benefit analyses.
import os
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

file0 = r"outputs/derived/mobility_part_0_with_network_distance.csv"
file1 = r"outputs/derived/mobility_part_1_with_network_distance.csv"
FILES = [file0, file1]

DIST_COL = "network_distance_m"
STATUS_COL = "route_status"
PARK_CLASS_COL = "park_class"
Y_COL = "steps"

CHUNKSIZE = 300_000
SEED = 123

JPY_PER_STEP = 0.04
POP_SCALE = 134.38
# Monetary benefits combine the per-step valuation with the population expansion factor.
MONEY_MULT = JPY_PER_STEP * POP_SCALE
MONEY_UNIT = 1_000_000.0

TYPE_NAME = {
    "A": "Block Park",
    "B": "Neighborhood Park",
    "C": "District Park",
    "D": "Comprehensive Park",
    "E": "Regional Park",
}
LABEL_MAP = {
    "A": "A. Block Park",
    "B": "B. Neighborhood Park",
    "C": "C. District Park",
    "D": "D. Comprehensive Park",
    "E": "E. Regional Park",
}

TEXT_SCALE = 1.2
plt.rcParams.update({
    "font.size":        10 * TEXT_SCALE,
    "axes.labelsize":   10 * TEXT_SCALE,
    "xtick.labelsize":   9 * TEXT_SCALE,
    "ytick.labelsize":   9 * TEXT_SCALE,
    "legend.fontsize":   8.5 * TEXT_SCALE,
})

def ols_loglog(x, y):
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[m]; y = y[m]
    if x.size < 5:
        return np.nan, np.nan, np.nan

    lx = np.log(x); ly = np.log(y)
    n = lx.size
    sx = lx.sum(); sy = ly.sum()
    sxx = (lx * lx).sum()
    sxy = (lx * ly).sum()
    denom = (n * sxx - sx * sx)
    if denom == 0:
        return np.nan, np.nan, np.nan

    beta = (n * sxy - sx * sy) / denom
    alpha = (sy - beta * sx) / n

    yhat = alpha + beta * lx
    ss_res = ((ly - yhat) ** 2).sum()
    ss_tot = ((ly - ly.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(alpha), float(beta), float(r2)

def binned_median_trend(dist, y, n_bins=60, min_bin_n=200):
    d = np.asarray(dist, np.float64)
    v = np.asarray(y, np.float64)
    m = np.isfinite(d) & np.isfinite(v) & (d > 0) & (v > 0)
    d = d[m]; v = v[m]
    if d.size < 1000:
        return pd.DataFrame(columns=["bin", "n", "dist_med", "y_med"])

    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(d, qs)
    edges = np.unique(edges)
    if edges.size < 10:
        edges = np.linspace(d.min(), d.max(), n_bins + 1)

    bin_id = pd.cut(d, bins=edges, include_lowest=True, labels=False)
    tmp = pd.DataFrame({"bin": bin_id, "dist": d, "y": v})
    g = (tmp.groupby("bin", as_index=False)
             .agg(n=("y", "size"),
                  dist_med=("dist", "median"),
                  y_med=("y", "median")))
    g = g[g["n"] >= min_bin_n].copy()
    g = g.sort_values("dist_med")
    return g

def quantile_binned_median(d, y, q_bins=100):
    d = np.asarray(d, np.float64)
    y = np.asarray(y, np.float64)
    n = len(d)
    if n < max(400, q_bins * 8):
        return None
    qs = np.linspace(0, 1, q_bins + 1)
    edges = np.quantile(d, qs)
    edges = np.unique(edges)
    if len(edges) < 5:
        return None
    bid = np.searchsorted(edges, d, side="right") - 1
    ok = (bid >= 0) & (bid < len(edges) - 1)
    d = d[ok]; y = y[ok]; bid = bid[ok]
    dfb = pd.DataFrame({"bin": bid, "d": d, "y": y})
    agg = dfb.groupby("bin").agg(d_med=("d", "median"), y_med=("y", "median"), n=("y", "size")).reset_index()
    return agg.sort_values("d_med")

def loess_np(x, y, frac=0.5, x_new=None):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    idx = np.argsort(x)
    x = x[idx]; y = y[idx]
    n = len(x)
    if n < 3:
        return x, y
    span = int(np.ceil(frac * n))
    span = max(3, min(span, n))
    if x_new is None:
        x_new = np.linspace(x.min(), x.max(), 450)
    else:
        x_new = np.asarray(x_new, dtype=np.float64)
    y_new = np.empty_like(x_new)

    for i, xi in enumerate(x_new):
        dist = np.abs(x - xi)
        nn_idx = np.argpartition(dist, span - 1)[:span]
        xk = x[nn_idx]; yk = y[nn_idx]; dk = dist[nn_idx]
        dmax = dk.max()
        if dmax <= 0:
            y_new[i] = yk.mean()
            continue
        u = dk / dmax
        w = (1 - u**3)**3
        X = np.column_stack([np.ones_like(xk), xk - xi])
        XtW = X.T * w
        A = XtW @ X
        b = XtW @ yk
        A[0, 0] += 1e-12
        A[1, 1] += 1e-12
        beta = np.linalg.solve(A, b)
        y_new[i] = beta[0]
    return x_new, y_new

def make_fixed_quantile_edges(d, q_bins):
    d = np.asarray(d, np.float64)
    d = d[np.isfinite(d) & (d > 0)]
    if d.size < 1000:
        return None
    edges = np.quantile(d, np.linspace(0, 1, q_bins + 1))
    edges = np.unique(edges)
    if edges.size < 5:
        edges = np.linspace(d.min(), d.max(), q_bins + 1)
    return edges

def binned_median_with_edges(d, y, edges, min_bin_n=200):
    d = np.asarray(d, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(d) & np.isfinite(y) & (d > 0) & (y > 0)
    d = d[m]; y = y[m]
    if d.size == 0:
        return None
    bid = pd.cut(d, bins=edges, include_lowest=True, labels=False)
    tmp = pd.DataFrame({"bin": bid, "d": d, "y": y})
    agg = (tmp.groupby("bin", as_index=False)
              .agg(d_med=("d", "median"), y_med=("y", "median"), n=("y", "size")))
    agg = agg[agg["n"] >= min_bin_n].copy()
    agg = agg.sort_values("d_med")
    return agg if len(agg) else None

def smooth_series_nan(y, win=11, min_valid=6):
    s = pd.Series(y, dtype="float64")
    cnt = s.rolling(win, center=True, min_periods=1).count()
    sm = s.rolling(win, center=True, min_periods=1).mean()
    sm[cnt < min_valid] = np.nan
    return sm.to_numpy()

def first_crossing_x(x, y, y0):
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if x.size < 2:
        return np.nan, False
    for i in range(1, x.size):
        if (y[i-1] >= y0) and (y[i] < y0):
            x1, x2 = x[i-1], x[i]
            y1, y2 = y[i-1], y[i]
            if y2 == y1:
                return x2, True
            t = (y0 - y1) / (y2 - y1)
            return x1 + t * (x2 - x1), True
    return np.nan, False

def stream_sample_network(files, usecols, chunksize=300_000, target_sample=200_000, seed=123):
    rng = np.random.default_rng(seed)
    parts = []
    kept = 0

    for fp in files:
        total_size = os.path.getsize(fp)
        pbar = tqdm(total=total_size, desc=f"Sampling {os.path.basename(fp)}", unit="B", unit_scale=True)

        for chunk in pd.read_csv(fp, usecols=usecols, chunksize=chunksize):
            pbar.update(int(chunk.memory_usage(deep=True).sum()))
            chunk = chunk.dropna(subset=usecols)

            chunk = chunk[
                (chunk[STATUS_COL] == "ok") &
                (chunk[DIST_COL] > 0) &
                (chunk[Y_COL] > 0)
            ]
            if len(chunk) == 0:
                continue

            remaining = target_sample - kept
            if remaining <= 0:
                break

            frac = min(1.0, remaining / max(len(chunk), 1))
            take = chunk.sample(frac=frac, random_state=int(rng.integers(0, 1e9))) if frac < 1.0 else chunk

            parts.append(take.astype({
                DIST_COL: "float64",
                Y_COL: "float64"
            }))
            kept += len(take)

        pbar.close()

    df = pd.concat(parts, ignore_index=True)
    if len(df) > target_sample:
        df = df.sample(n=target_sample, random_state=seed)
    return df

def read_network_full(files, usecols):
    dfs = []
    for fp in files:
        tmp = pd.read_csv(fp, usecols=usecols, low_memory=False)
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)
    df = df.dropna(subset=usecols).copy()
    df = df[(df[STATUS_COL] == "ok") & (df[DIST_COL] > 0) & (df[Y_COL] > 0)]
    df[PARK_CLASS_COL] = df[PARK_CLASS_COL].astype(str).str.strip()
    return df


In [ ]:
# Estimate and plot the distance-decay relationship for health benefits.
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

file0 = r"outputs/derived/mobility_part_0_with_network_distance.csv"
file1 = r"outputs/derived/mobility_part_1_with_network_distance.csv"
FILES = [file0, file1]

DIST_COL = "network_distance_m"
STATUS_COL = "route_status"
Y_COL = "steps"

CHUNKSIZE = 300_000
SEED = 123

JPY_PER_STEP = 0.04
POP_SCALE = 134.38
MONEY_MULT = JPY_PER_STEP * POP_SCALE
MONEY_UNIT = 1_000_000.0

TARGET_SAMPLE_A = 200_000
N_BINS_A = 60
MIN_BIN_N_A = 200

TEXT_SCALE = 1.0
plt.rcParams.update({
    "font.size":        11 * TEXT_SCALE,
    "axes.labelsize":   11 * TEXT_SCALE,
    "xtick.labelsize":  10 * TEXT_SCALE,
    "ytick.labelsize":  10 * TEXT_SCALE,
    "legend.fontsize":   9 * TEXT_SCALE,
})

def ols_loglog(x, y):
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[m]
    y = y[m]
    if x.size < 5:
        return np.nan, np.nan, np.nan

    lx = np.log(x)
    ly = np.log(y)

    n = lx.size
    sx = lx.sum()
    sy = ly.sum()
    sxx = (lx * lx).sum()
    sxy = (lx * ly).sum()
    denom = (n * sxx - sx * sx)

    if denom == 0:
        return np.nan, np.nan, np.nan

    beta = (n * sxy - sx * sy) / denom
    alpha = (sy - beta * sx) / n

    yhat = alpha + beta * lx
    ss_res = ((ly - yhat) ** 2).sum()
    ss_tot = ((ly - ly.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return float(alpha), float(beta), float(r2)

def binned_median_trend(dist, y, n_bins=60, min_bin_n=200):
    d = np.asarray(dist, np.float64)
    v = np.asarray(y, np.float64)

    m = np.isfinite(d) & np.isfinite(v) & (d > 0) & (v > 0)
    d = d[m]
    v = v[m]

    if d.size < 1000:
        return pd.DataFrame(columns=["bin", "n", "dist_med", "y_med"])

    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(d, qs)
    edges = np.unique(edges)

    if edges.size < 10:
        edges = np.linspace(d.min(), d.max(), n_bins + 1)

    bin_id = pd.cut(d, bins=edges, include_lowest=True, labels=False)
    tmp = pd.DataFrame({"bin": bin_id, "dist": d, "y": v})

    g = (
        tmp.groupby("bin", as_index=False)
           .agg(n=("y", "size"),
                dist_med=("dist", "median"),
                y_med=("y", "median"))
    )

    g = g[g["n"] >= min_bin_n].copy()
    g = g.sort_values("dist_med")
    return g

def stream_sample_network(files, usecols, chunksize=300_000, target_sample=200_000, seed=123):
    rng = np.random.default_rng(seed)
    parts = []
    kept = 0

    for fp in files:
        total_size = os.path.getsize(fp)
        pbar = tqdm(total=total_size, desc=f"Sampling {os.path.basename(fp)}", unit="B", unit_scale=True)

        for chunk in pd.read_csv(fp, usecols=usecols, chunksize=chunksize):
            pbar.update(int(chunk.memory_usage(deep=True).sum()))

            chunk = chunk.dropna(subset=usecols)
            chunk = chunk[
                (chunk[STATUS_COL] == "ok") &
                (chunk[DIST_COL] > 0) &
                (chunk[Y_COL] > 0)
            ]

            if len(chunk) == 0:
                continue

            remaining = target_sample - kept
            if remaining <= 0:
                break

            frac = min(1.0, remaining / max(len(chunk), 1))
            if frac < 1.0:
                take = chunk.sample(frac=frac, random_state=int(rng.integers(0, 1e9)))
            else:
                take = chunk

            parts.append(take.astype({
                DIST_COL: "float64",
                Y_COL: "float64"
            }))
            kept += len(take)

        pbar.close()

    df = pd.concat(parts, ignore_index=True)
    if len(df) > target_sample:
        df = df.sample(n=target_sample, random_state=seed)

    return df

USECOLS_A = [DIST_COL, STATUS_COL, Y_COL]

df_a = stream_sample_network(
    FILES,
    usecols=USECOLS_A,
    chunksize=CHUNKSIZE,
    target_sample=TARGET_SAMPLE_A,
    seed=SEED
)

df_a["money_mjpy"] = (df_a[Y_COL] * MONEY_MULT) / MONEY_UNIT

YMAX_STEPS = 600_000
YMAX = (YMAX_STEPS * MONEY_MULT) / MONEY_UNIT
clip_ratio = float((df_a["money_mjpy"] > YMAX).mean())

trend_all = binned_median_trend(
    df_a[DIST_COL].to_numpy(),
    df_a["money_mjpy"].to_numpy(),
    n_bins=N_BINS_A,
    min_bin_n=MIN_BIN_N_A
)

alpha_all, beta_all, r2_all = ols_loglog(
    df_a[DIST_COL].to_numpy(),
    df_a["money_mjpy"].to_numpy()
)

if len(trend_all) > 0:
    alpha_bin, beta_bin, r2_bin = ols_loglog(
        trend_all["dist_med"].to_numpy(),
        trend_all["y_med"].to_numpy()
    )
else:
    alpha_bin, beta_bin, r2_bin = np.nan, np.nan, np.nan

sigma_all = np.exp(alpha_all) if np.isfinite(alpha_all) else np.nan
sigma_bin = np.exp(alpha_bin) if np.isfinite(alpha_bin) else np.nan

dpos = df_a.loc[df_a[DIST_COL] > 0, DIST_COL]
x_min = float(dpos.min()) if len(dpos) else 1.0
x_max = 10000

x_grid = np.linspace(x_min, x_max, 300)
y_fit_all = sigma_all * (x_grid ** beta_all) if (np.isfinite(sigma_all) and np.isfinite(beta_all)) else None

fig, ax = plt.subplots(figsize=(8.8, 4.8))

ax.scatter(
    df_a[DIST_COL],
    df_a["money_mjpy"],
    s=1,
    alpha=0.03,
    rasterized=True
)

if len(trend_all) > 0:
    ax.plot(
        trend_all["dist_med"],
        trend_all["y_med"],
        linewidth=2.0,
        label=f"Binned median trend (bins={len(trend_all)})"
    )

if y_fit_all is not None:
    ax.plot(
        x_grid,
        y_fit_all,
        linewidth=1.8,
        linestyle="--",
        label="Power-law fit (log-log OLS on sample)"
    )

ax.set_xlim(0, x_max)
ax.set_ylim(0, YMAX)
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Annual health benefit (million JPY)")

fmt = ScalarFormatter(useOffset=False)
fmt.set_scientific(False)
ax.yaxis.set_major_formatter(fmt)
ax.get_yaxis().get_offset_text().set_visible(False)

ax.margins(x=0, y=0)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.35)

ax.legend(loc="upper left", frameon=True)

if len(trend_all) > 0 and np.isfinite(alpha_bin) and np.isfinite(beta_bin):
    axins = inset_axes(
        ax,
        width="40%", height="37%",
        loc="upper right",
        borderpad=1.6
    )

    axins.scatter(
        trend_all["dist_med"],
        trend_all["y_med"],
        s=12,
        alpha=0.8
    )

    x2 = np.linspace(trend_all["dist_med"].min(), trend_all["dist_med"].max(), 200)
    y2 = sigma_bin * (x2 ** beta_bin)
    axins.plot(x2, y2, linewidth=2.0)

    axins.set_xscale("log")
    axins.set_yscale("log")
    axins.tick_params(labelsize=8)
    axins.grid(True, linestyle="--", linewidth=0.4, alpha=0.35)

plt.tight_layout()
out_png = os.path.join("outputs", "fig2a_network_scatter_linear_millionJPY_with_binned_trend_inset_reproportioned.png")
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

print("Saved:", os.path.abspath(out_png))
print("clipped% =", clip_ratio * 100)
print(f"[Sample all points] ln(sigma)={alpha_all:.6f}  sigma={sigma_all:.6g}  beta={beta_all:.6f}  R2={r2_all:.4f}")
if len(trend_all) > 0:
    print(f"[Binned medians] ln(sigma)={alpha_bin:.6f}  sigma={sigma_bin:.6g}  beta={beta_bin:.6f}  R2={r2_bin:.4f}  bins_used={len(trend_all)}")


In [ ]:
# Evaluate the stability of distance-decay estimates across distance cutoffs.
USECOLS_B = [DIST_COL, STATUS_COL, PARK_CLASS_COL, Y_COL]

MAX_R2_DIST = 10_000
R2_GRID_N = 320
Q_BINS_R2 = 10
MIN_BIN_N = 200
MIN_BINS_FOR_FIT = 6
MIN_POINTS_FOR_FIT = 3000
R2_HLINE = 0.7
SMOOTH_WIN = 11
SMOOTH_MIN_VALID = 6

df_b = read_network_full(FILES, usecols=USECOLS_B)
df_b["money_mjpy"] = (df_b[Y_COL].astype(float) * MONEY_MULT) / MONEY_UNIT

groups = [g for g in list("ABCDE") if g in set(df_b[PARK_CLASS_COL].unique())]
print("[INFO] groups:", groups)

dmax_grid = np.linspace(300, MAX_R2_DIST, R2_GRID_N)

fig, ax = plt.subplots(figsize=(7.2, 8.0))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

LW = 1 * TEXT_SCALE
DOT_S = 40 * (TEXT_SCALE ** 2)

cross_summary = []

for g in groups:
    sub = df_b[df_b[PARK_CLASS_COL] == g]
    label = LABEL_MAP.get(g, str(g))

    d_all = sub[DIST_COL].to_numpy(np.float64)
    y_all = sub["money_mjpy"].to_numpy(np.float64)

    m = np.isfinite(d_all) & np.isfinite(y_all) & (d_all > 0) & (y_all > 0)
    d_all = d_all[m]; y_all = y_all[m]

    if d_all.size < MIN_POINTS_FOR_FIT:
        ax.plot(dmax_grid, [np.nan]*len(dmax_grid), lw=LW, label=label)
        cross_summary.append((label, np.nan))
        continue

    order = np.argsort(d_all)
    d_all = d_all[order]
    y_all = y_all[order]

    edges = make_fixed_quantile_edges(d_all[d_all <= MAX_R2_DIST], Q_BINS_R2)
    if edges is None:
        ax.plot(dmax_grid, [np.nan]*len(dmax_grid), lw=LW, label=label)
        cross_summary.append((label, np.nan))
        continue

    r2s = []
    for dm in dmax_grid:
        idx = np.searchsorted(d_all, dm, side="right")
        if idx < MIN_POINTS_FOR_FIT:
            r2s.append(np.nan)
            continue

        agg = binned_median_with_edges(d_all[:idx], y_all[:idx], edges, min_bin_n=MIN_BIN_N)
        if agg is None or len(agg) < MIN_BINS_FOR_FIT:
            r2s.append(np.nan)
            continue

        _, _, r2 = ols_loglog(agg["d_med"].to_numpy(), agg["y_med"].to_numpy())
        r2s.append(r2)

    r2s = np.asarray(r2s, np.float64)
    r2s_sm = smooth_series_nan(r2s, win=SMOOTH_WIN, min_valid=SMOOTH_MIN_VALID)

    ax.plot(dmax_grid, r2s_sm, lw=LW, label=label)

    if R2_HLINE is not None:
        xc, ok = first_crossing_x(dmax_grid, r2s_sm, R2_HLINE)
        if ok:
            ax.scatter([xc], [R2_HLINE], s=DOT_S, zorder=6)
            cross_summary.append((label, xc))
        else:
            cross_summary.append((label, np.nan))

ax.set_xlabel("Max network distance (m)")
ax.set_ylabel(r"$R^2$ (log–log OLS" + "\n" + r"on binned medians)")
ax.set_xlim(0, MAX_R2_DIST)
ax.set_ylim(0.4, 1.0)
ax.grid(True, linestyle="--", linewidth=0.6 * TEXT_SCALE, alpha=0.35)

if R2_HLINE is not None:
    ax.axhline(R2_HLINE, linestyle="--", linewidth=1.1 * TEXT_SCALE, alpha=0.85)
    ax.text(MAX_R2_DIST * 0.995, R2_HLINE - 0.012, f"{R2_HLINE:.1f}",
            ha="right", va="top")

leg = ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.02, 0.02),
    frameon=True,
    framealpha=0.85,
    borderaxespad=0.0,
)
leg.get_frame().set_facecolor("white")
leg.get_frame().set_edgecolor("none")

plt.tight_layout()
out2 = os.path.join("outputs", f"fig2b_network_r2_vs_dmax_x{MAX_R2_DIST}_smooth.png")
plt.savefig(out2, dpi=260, bbox_inches="tight")
plt.show()

print("Saved:", os.path.abspath(out2))
print("\n=== First crossing of R2 threshold (approx., based on smoothed curve) ===")
for lab, xc in cross_summary:
    if np.isfinite(xc):
        print(f"{lab:24s}: ~{xc:,.0f} m")
    else:
        print(f"{lab:24s}: (no crossing / insufficient data)")


In [ ]:
# Estimate park-type-specific distance-benefit relationships.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

file0 = r"outputs/derived/mobility_part_0_with_network_distance.csv"
file1 = r"outputs/derived/mobility_part_1_with_network_distance.csv"
FILES = [file0, file1]

DIST_COL = "network_distance_m"
STATUS_COL = "route_status"
PARK_CLASS_COL = "park_class"
STEP_COL = "steps"

JPY_PER_STEP = 0.04
POP_SCALE = 134.38
MONEY_MULT = JPY_PER_STEP * POP_SCALE
MONEY_UNIT = 1_000_000.0

XLINE = 250
XMAX_PLOT = 2000
YMAX_PLOT = 0.2
MIN_DIST_CLEAN = 10
Q_BINS_PLOT = 100
LOESS_FRAC = 0.5

FIT_MAX = 6000
FIT_BINS = 60
FIT_MIN_BIN_N = 30

TYPE_NAME = {
    "A": "Block Park",
    "B": "Neighborhood Park",
    "C": "District Park",
    "D": "Comprehensive Park",
    "E": "Regional Park",
}
LABEL_MAP = {
    "A": "A. Block Park",
    "B": "B. Neighborhood Park",
    "C": "C. District Park",
    "D": "D. Comprehensive Park",
    "E": "E. Regional Park",
}

TEXT_SCALE = 1.5
plt.rcParams.update({
    "font.size":        11 * TEXT_SCALE,
    "axes.labelsize":   11 * TEXT_SCALE,
    "xtick.labelsize":  10 * TEXT_SCALE,
    "ytick.labelsize":  10 * TEXT_SCALE,
    "legend.fontsize":   9 * TEXT_SCALE,
})

def quantile_binned_median(d, y, q_bins=100):
    d = np.asarray(d, np.float64)
    y = np.asarray(y, np.float64)

    m = np.isfinite(d) & np.isfinite(y) & (d > 0) & (y > 0)
    d = d[m]
    y = y[m]

    n = len(d)
    if n < max(400, q_bins * 8):
        return None

    qs = np.linspace(0, 1, q_bins + 1)
    edges = np.quantile(d, qs)
    edges = np.unique(edges)
    if len(edges) < 5:
        return None

    bid = np.searchsorted(edges, d, side="right") - 1
    ok = (bid >= 0) & (bid < len(edges) - 1)
    d = d[ok]
    y = y[ok]
    bid = bid[ok]

    dfb = pd.DataFrame({"bin": bid, "d": d, "y": y})
    agg = (
        dfb.groupby("bin")
           .agg(d_med=("d", "median"),
                y_med=("y", "median"),
                n=("y", "size"))
           .reset_index()
           .sort_values("d_med")
    )
    return agg

def loess_np(x, y, frac=0.5, x_new=None):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    idx = np.argsort(x)
    x = x[idx]
    y = y[idx]

    n = len(x)
    if n < 3:
        return x, y

    span = int(np.ceil(frac * n))
    span = max(3, min(span, n))

    if x_new is None:
        x_new = np.linspace(x.min(), x.max(), 450)
    else:
        x_new = np.asarray(x_new, dtype=np.float64)

    y_new = np.empty_like(x_new)

    for i, xi in enumerate(x_new):
        dist = np.abs(x - xi)
        nn_idx = np.argpartition(dist, span - 1)[:span]
        xk = x[nn_idx]
        yk = y[nn_idx]
        dk = dist[nn_idx]

        dmax = dk.max()
        if dmax <= 0:
            y_new[i] = yk.mean()
            continue

        u = dk / dmax
        w = (1 - u**3)**3

        X = np.column_stack([np.ones_like(xk), xk - xi])
        XtW = X.T * w
        A = XtW @ X
        b = XtW @ yk

        A[0, 0] += 1e-12
        A[1, 1] += 1e-12

        beta = np.linalg.solve(A, b)
        y_new[i] = beta[0]

    return x_new, y_new

def binned_median_for_fit(d, y, n_bins=60, min_bin_n=30):
    d = np.asarray(d, np.float64)
    y = np.asarray(y, np.float64)

    m = np.isfinite(d) & np.isfinite(y) & (d > 0) & (y > 0)
    d = d[m]
    y = y[m]

    if len(d) < max(300, n_bins * 5):
        return None

    qs = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(d, qs)
    edges = np.unique(edges)

    if len(edges) < 5:
        return None

    bid = pd.cut(d, bins=edges, include_lowest=True, labels=False)
    tmp = pd.DataFrame({"bin": bid, "d": d, "y": y})

    agg = (
        tmp.groupby("bin", as_index=False)
           .agg(d_med=("d", "median"),
                y_med=("y", "median"),
                n=("y", "size"))
    )

    agg = agg[agg["n"] >= min_bin_n].copy()
    agg = agg.sort_values("d_med")

    if len(agg) < 5:
        return None

    return agg

def ols_loglog(x, y):
    x = np.asarray(x, np.float64)
    y = np.asarray(y, np.float64)

    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[m]
    y = y[m]

    if x.size < 5:
        return np.nan, np.nan, np.nan

    lx = np.log(x)
    ly = np.log(y)

    n = lx.size
    sx = lx.sum()
    sy = ly.sum()
    sxx = (lx * lx).sum()
    sxy = (lx * ly).sum()

    den = n * sxx - sx * sx
    if den == 0:
        return np.nan, np.nan, np.nan

    beta = (n * sxy - sx * sy) / den
    alpha = (sy - beta * sx) / n

    yhat = alpha + beta * lx
    ss_res = ((ly - yhat) ** 2).sum()
    ss_tot = ((ly - ly.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return float(alpha), float(beta), float(r2)

dfs = [pd.read_csv(f, low_memory=False) for f in FILES]
df_plot = pd.concat(dfs, ignore_index=True)

df_plot = df_plot.dropna(subset=[STEP_COL, PARK_CLASS_COL, DIST_COL, STATUS_COL]).copy()
df_plot = df_plot[
    (df_plot[STATUS_COL] == "ok") &
    (df_plot[STEP_COL] > 0) &
    (df_plot[DIST_COL] >= MIN_DIST_CLEAN)
].copy()

df_plot[PARK_CLASS_COL] = df_plot[PARK_CLASS_COL].astype(str).str.strip()

df_plot["money_mjpy"] = df_plot[STEP_COL].astype(float) * MONEY_MULT / MONEY_UNIT

print("Routed subset N =", len(df_plot))
print("Median steps =", df_plot[STEP_COL].median())
print("Median annual benefit (million JPY) =", df_plot["money_mjpy"].median())

groups = [g for g in list("ABCDE") if g in set(df_plot[PARK_CLASS_COL].unique())]

Y_LABEL_2LINE = "Annual health benefit\n(million JPY)"

fig, ax = plt.subplots(figsize=(7.2, 8.0))
LW_MAIN  = 2.2
LW_VLINE = 1.2

param_rows = []

for g in groups:
    sub = df_plot[df_plot[PARK_CLASS_COL] == g].copy()

    sub_plot = sub[sub[DIST_COL] <= XMAX_PLOT].copy()
    d_plot = sub_plot[DIST_COL].to_numpy()
    y_plot = sub_plot["money_mjpy"].to_numpy()

    if len(d_plot) >= 2000:
        agg_plot = quantile_binned_median(d_plot, y_plot, q_bins=Q_BINS_PLOT)
        if agg_plot is not None and len(agg_plot) >= 12:
            x_med = agg_plot["d_med"].to_numpy(np.float64)
            y_med = agg_plot["y_med"].to_numpy(np.float64)
            xl, yl = loess_np(x_med, y_med, frac=LOESS_FRAC)
            ax.plot(xl, yl, linewidth=LW_MAIN, label=LABEL_MAP.get(g, g))

    sub_fit = sub[sub[DIST_COL] <= FIT_MAX].copy()
    agg_fit = binned_median_for_fit(
        sub_fit[DIST_COL].to_numpy(),
        sub_fit["money_mjpy"].to_numpy(),
        n_bins=FIT_BINS,
        min_bin_n=FIT_MIN_BIN_N
    )

    if agg_fit is None:
        alpha, beta, r2 = np.nan, np.nan, np.nan
        n_fit = len(sub_fit)
        bins_used = 0
    else:
        alpha, beta, r2 = ols_loglog(
            agg_fit["d_med"].to_numpy(),
            agg_fit["y_med"].to_numpy()
        )
        n_fit = len(sub_fit)
        bins_used = len(agg_fit)

    param_rows.append({
        "Type": g,
        "Park type name": TYPE_NAME.get(g, ""),
        "N_raw_within_6000m": n_fit,
        "Bins_used": bins_used,
        "loge_sigma": alpha,
        "sigma": np.exp(alpha) if np.isfinite(alpha) else np.nan,
        "beta": beta,
        "R2": r2
    })

ax.axvline(XLINE, linestyle="--", linewidth=LW_VLINE, alpha=0.8)

ax.set_xlabel("Pedestrian network distance (m)")
ax.set_ylabel(Y_LABEL_2LINE)
ax.set_xlim(0, XMAX_PLOT)
ax.set_ylim(0, YMAX_PLOT)

ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.4)

fmt = ScalarFormatter(useOffset=False)
fmt.set_scientific(False)
ax.yaxis.set_major_formatter(fmt)
ax.get_yaxis().get_offset_text().set_visible(False)

leg = ax.legend(
    loc="upper right",
    bbox_to_anchor=(0.98, 0.99),
    fontsize=14,
    frameon=True, framealpha=0.90,
    borderaxespad=0.0
)
leg.get_frame().set_edgecolor("none")

plt.tight_layout()
out_fig = os.path.join("outputs", "fig2c_network_loess_by_type_updated_ymax0.2.png")
plt.savefig(out_fig, dpi=260, bbox_inches="tight")
plt.show()
print("Saved figure:", os.path.abspath(out_fig))

param_df = pd.DataFrame(param_rows).sort_values("Type")

print("\n=== Type-specific power-law parameters using network distance ===")
display(param_df)

out_table = os.path.join("outputs", "fig2c_network_type_equation_params_updated_ymax0.2.csv")
param_df.to_csv(out_table, index=False, encoding="utf-8-sig")
print("Saved table:", os.path.abspath(out_table))


In [ ]:
# Summarize access-walking time relative to time spent inside parks.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file0 = Path("outputs/derived/mobility_part_0_with_network_distance.csv")
file1 = Path("outputs/derived/mobility_part_1_with_network_distance.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WALK_SPEED_MPS = 1.2
DIST_MAX_M = 1500
USE_ROUTE_STATUS = "ok"

BINS = [0, 0.1, 0.25, 0.5, 1, np.inf]
BIN_LABELS = ["<0.10", "0.10–0.25", "0.25–0.50", "0.50–1.00", "≥1.00"]

TYPE_NAME = {
    "A": "Block Park",
    "B": "Neighborhood Park",
    "C": "District Park",
    "D": "Comprehensive Park",
    "E": "Regional Park",
}

df0 = pd.read_csv(file0, low_memory=False)
df1 = pd.read_csv(file1, low_memory=False)

OLD_ANALYSIS_COLS = [
    "access_time_oneway_s",
    "access_time_roundtrip_s",
    "time_ratio_oneway",
    "time_ratio_roundtrip",
    "distance_group",
    "walk_access_flag",
    "inpark_active_time_s",
    "benefit_ratio_oneway",
    "benefit_ratio_roundtrip"
]

df0 = df0.drop(columns=[c for c in OLD_ANALYSIS_COLS if c in df0.columns], errors="ignore")
df1 = df1.drop(columns=[c for c in OLD_ANALYSIS_COLS if c in df1.columns], errors="ignore")

df = pd.concat([df0, df1], ignore_index=True)

print("Rows after concatenation:", len(df))
print("Columns:", df.columns.tolist())

need_cols = [
    "user_ID", "osm_id",
    "visit_count", "delta_time_total",
    "network_distance_m", "route_status", "park_class"
]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for c in ["visit_count", "delta_time_total", "network_distance_m"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["park_class"] = df["park_class"].astype(str).str.strip()

df_use = df[
    (df["route_status"] == USE_ROUTE_STATUS) &
    df["visit_count"].notna() &
    df["delta_time_total"].notna() &
    df["network_distance_m"].notna() &
    (df["visit_count"] > 0) &
    (df["delta_time_total"] > 0) &
    (df["network_distance_m"] > 0) &
    (df["network_distance_m"] < DIST_MAX_M)
].copy()

print("\nRows retained for analysis:", len(df_use))
print("Median network distance (m):", df_use["network_distance_m"].median())
print("Median visit count:", df_use["visit_count"].median())
print("Median delta_time_total (s):", df_use["delta_time_total"].median())

df_use["access_time_oneway_s"] = (
    df_use["visit_count"] * df_use["network_distance_m"] / WALK_SPEED_MPS
)

df_use["access_time_roundtrip_s"] = 2.0 * df_use["access_time_oneway_s"]

df_use["time_ratio_oneway"] = df_use["access_time_oneway_s"] / df_use["delta_time_total"]
df_use["time_ratio_roundtrip"] = df_use["access_time_roundtrip_s"] / df_use["delta_time_total"]

def summarize_ratio(series, prefix):
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s) & (s >= 0)]

    if len(s) == 0:
        return pd.Series({
            f"{prefix}_n": 0,
            f"{prefix}_median": np.nan,
            f"{prefix}_q1": np.nan,
            f"{prefix}_q3": np.nan,
            f"{prefix}_p90": np.nan,
            f"{prefix}_mean": np.nan,
            f"{prefix}_share_lt_0.10": np.nan,
            f"{prefix}_share_lt_0.25": np.nan,
            f"{prefix}_share_lt_0.50": np.nan,
            f"{prefix}_share_lt_1.00": np.nan,
            f"{prefix}_share_ge_1.00": np.nan,
        })

    return pd.Series({
        f"{prefix}_n": len(s),
        f"{prefix}_median": s.median(),
        f"{prefix}_q1": s.quantile(0.25),
        f"{prefix}_q3": s.quantile(0.75),
        f"{prefix}_p90": s.quantile(0.90),
        f"{prefix}_mean": s.mean(),
        f"{prefix}_share_lt_0.10": (s < 0.10).mean(),
        f"{prefix}_share_lt_0.25": (s < 0.25).mean(),
        f"{prefix}_share_lt_0.50": (s < 0.50).mean(),
        f"{prefix}_share_lt_1.00": (s < 1.00).mean(),
        f"{prefix}_share_ge_1.00": (s >= 1.00).mean(),
    })

def bin_shares(series, bins, labels):
    s = pd.to_numeric(series, errors="coerce")
    s = s[np.isfinite(s) & (s >= 0)]
    cats = pd.cut(s, bins=bins, labels=labels, right=False, include_lowest=True)
    out = cats.value_counts(normalize=True).reindex(labels, fill_value=0.0)
    return out

summary_overall = pd.concat([
    summarize_ratio(df_use["time_ratio_oneway"], "time_ratio_oneway"),
    summarize_ratio(df_use["time_ratio_roundtrip"], "time_ratio_roundtrip"),
])

summary_overall = summary_overall.to_frame(name="value").reset_index().rename(columns={"index": "metric"})

print("\n=== Overall summary ===")
display(summary_overall)

by_class_rows = []
for g in sorted(df_use["park_class"].dropna().unique()):
    sub = df_use[df_use["park_class"] == g].copy()

    row = {"park_class": g, "park_class_name": TYPE_NAME.get(g, g)}
    row.update(summarize_ratio(sub["time_ratio_oneway"], "time_ratio_oneway").to_dict())
    row.update(summarize_ratio(sub["time_ratio_roundtrip"], "time_ratio_roundtrip").to_dict())
    by_class_rows.append(row)

summary_by_class = pd.DataFrame(by_class_rows)

print("\n=== By park_class summary ===")
display(summary_by_class)

shares_oneway = bin_shares(df_use["time_ratio_oneway"], BINS, BIN_LABELS)
shares_roundtrip = bin_shares(df_use["time_ratio_roundtrip"], BINS, BIN_LABELS)

print("\n=== Share by ratio bin: one-way ===")
print(shares_oneway)

print("\n=== Share by ratio bin: round-trip ===")
print(shares_roundtrip)

colors = ["#d9f0a3", "#addd8e", "#78c679", "#31a354", "#006837"]

fig, ax = plt.subplots(figsize=(10, 2.8))

y_positions = [1, 0]
labels = ["One-way", "Round-trip"]
share_table = [shares_oneway, shares_roundtrip]

for y, lab, shares in zip(y_positions, labels, share_table):
    left = 0
    for color, bin_lab in zip(colors, BIN_LABELS):
        width = shares[bin_lab]
        ax.barh(y, width, left=left, height=0.35, color=color, edgecolor="white")
        if width >= 0.04:
            ax.text(left + width / 2, y, f"{width*100:.1f}%", ha="center", va="center", fontsize=10)
        left += width

ax.set_xlim(0, 1)
ax.set_yticks(y_positions)
ax.set_yticklabels(labels)
ax.set_xlabel("Share of user–park pairs")
ax.set_title(
    f"Home-to-park access time relative to full in-park dwell time\n"
    f"(network distance < {DIST_MAX_M} m, walking speed = {WALK_SPEED_MPS:.1f} m/s)"
)

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, edgecolor="white", label=l) for c, l in zip(colors, BIN_LABELS)]
ax.legend(handles=legend_handles, title="Time ratio bin", ncol=5, bbox_to_anchor=(0.5, -0.25), loc="upper center")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

fig_path = OUTPUT_DIR / "access_vs_inpark_time_ratio_full_dwell_lt1500m_oneway_roundtrip.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nFigure saved:", fig_path)

pair_out = OUTPUT_DIR / "user_park_access_inpark_time_ratio_pair_level_full_dwell_lt1500m.csv"
overall_out = OUTPUT_DIR / "user_park_access_inpark_time_ratio_summary_overall_full_dwell_lt1500m.csv"
class_out = OUTPUT_DIR / "user_park_access_inpark_time_ratio_summary_by_park_class_full_dwell_lt1500m.csv"

pair_cols = [
    "user_ID", "osm_id", "park_class",
    "visit_count", "delta_time_total",
    "network_distance_m", "route_status",
    "access_time_oneway_s", "access_time_roundtrip_s",
    "time_ratio_oneway", "time_ratio_roundtrip"
]
pair_cols = [c for c in pair_cols if c in df_use.columns]

df_use[pair_cols].to_csv(pair_out, index=False, encoding="utf-8-sig")
summary_overall.to_csv(overall_out, index=False, encoding="utf-8-sig")
summary_by_class.to_csv(class_out, index=False, encoding="utf-8-sig")

print("Detailed results saved:", pair_out)
print("Overall summary saved:", overall_out)
print("Park-type summary saved:", class_out)


In [ ]:
# Estimate the benefit increment associated with hypothetical access walking.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

file0 = Path("outputs/derived/mobility_part_0_with_network_distance.csv")
file1 = Path("outputs/derived/mobility_part_1_with_network_distance.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIST_MAX_M = 1500
WALK_SPEED_MPS = 1.2
CADENCE = 1.2
JPY_PER_STEP = 0.04
POP_SCALE = 134.38
USE_ROUTE_STATUS = "ok"

df0 = pd.read_csv(file0, low_memory=False)
df1 = pd.read_csv(file1, low_memory=False)
df = pd.concat([df0, df1], ignore_index=True)

print("Total rows:", len(df))

need_cols = ["steps", "visit_count", "network_distance_m", "route_status"]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for c in ["steps", "visit_count", "network_distance_m"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["baseline_inpark_benefit_jpy"] = df["steps"] * JPY_PER_STEP * POP_SCALE

near_mask = (
    (df["route_status"] == USE_ROUTE_STATUS) &
    df["visit_count"].notna() &
    df["network_distance_m"].notna() &
    (df["visit_count"] > 0) &
    (df["network_distance_m"] > 0) &
    (df["network_distance_m"] < DIST_MAX_M)
)

df_near = df.loc[near_mask].copy()

print("Rows in the short-distance routed subset:", len(df_near))

# Access steps combine route distance, visit frequency, walking speed, and cadence.
df_near["access_steps_oneway"] = (
    df_near["visit_count"] * (df_near["network_distance_m"] / WALK_SPEED_MPS) * CADENCE
)

df_near["access_steps_roundtrip"] = 2.0 * df_near["access_steps_oneway"]

df_near["access_benefit_oneway_jpy"] = df_near["access_steps_oneway"] * JPY_PER_STEP * POP_SCALE
df_near["access_benefit_roundtrip_jpy"] = df_near["access_steps_roundtrip"] * JPY_PER_STEP * POP_SCALE

baseline_total_jpy = df.loc[
    df["baseline_inpark_benefit_jpy"].notna() & (df["baseline_inpark_benefit_jpy"] > 0),
    "baseline_inpark_benefit_jpy"
].sum()

added_oneway_jpy = df_near["access_benefit_oneway_jpy"].sum()
added_roundtrip_jpy = df_near["access_benefit_roundtrip_jpy"].sum()

increase_pct_oneway = added_oneway_jpy / baseline_total_jpy if baseline_total_jpy > 0 else np.nan
increase_pct_roundtrip = added_roundtrip_jpy / baseline_total_jpy if baseline_total_jpy > 0 else np.nan

print("\n=== Key result: percentage increase relative to original baseline ===")
print(f"One-way access walking would increase total monetized benefit by: {increase_pct_oneway:.2%}")
print(f"Round-trip access walking would increase total monetized benefit by: {increase_pct_roundtrip:.2%}")

labels = ["One-way\nupper bound", "Round-trip\nupper bound"]
values = [increase_pct_oneway * 100, increase_pct_roundtrip * 100]
colors = ["#74c476", "#238b45"]

fig, ax = plt.subplots(figsize=(6.8, 5.0))

bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.6)

for rect, val in zip(bars, values):
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height(),
        f"{val:.1f}%",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_ylabel("Increase relative to baseline (%)")
ax.set_title(
    f"Potential increase in total health benefit if adding\n"
    f"home-to-park walking for pairs with network distance < {DIST_MAX_M} m"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()

fig_path = OUTPUT_DIR / "benefit_increase_ratio_if_adding_access_walking_lt1500m.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nFigure saved:", fig_path)


In [ ]:
# Summarize catchment distance, visit shares, and benefit shares by park type.
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

FILE_1 = r"outputs/derived/mobility_part_0_with_network_distance.csv"
FILE_2 = r"outputs/derived/mobility_part_1_with_network_distance.csv"

OUTPUT_DIR = Path(
    r"outputs/catchment_pattern"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIST_COL = "network_distance_m"
STEPS_COL = "steps"
PARK_CLASS_COL = "park_class"
PARK_CLASS_NAME_COL = "park_class_name"

WEIGHT_COL = None

STEP_VALUE_JPY = 0.04056

BIN_EDGES = [0, 250, 500, 1000, 3000, 6000, np.inf]
BIN_LABELS = ["≤250 m", "250–500 m", "500 m–1 km", "1–3 km", "3–6 km", ">6 km"]

PARK_TYPE_ORDER = [
    "City block park",
    "Neighborhood Park",
    "District Park",
    "Comprehensive Park",
    "Regional Park",
]

VISIT_COLORS = ["#f2f2f2", "#d9d9d9", "#bdbdbd", "#969696", "#737373", "#525252"]
BENEFIT_COLORS = ["#edf8e9", "#c7e9c0", "#a1d99b", "#74c476", "#41ab5d", "#238b45"]

FIG_W = 12
FIG_H = 8
LABEL_MIN_SHARE = 0.045

df1 = pd.read_csv(FILE_1)
df2 = pd.read_csv(FILE_2)
df = pd.concat([df1, df2], ignore_index=True).copy()

required_cols = [DIST_COL, STEPS_COL, PARK_CLASS_COL, PARK_CLASS_NAME_COL]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")
df[STEPS_COL] = pd.to_numeric(df[STEPS_COL], errors="coerce")

if WEIGHT_COL is not None:
    if WEIGHT_COL not in df.columns:
        raise ValueError(f"WEIGHT_COL='{WEIGHT_COL}' not found in dataframe.")
    df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors="coerce")
else:
    df["__weight__"] = 1.0
    WEIGHT_COL = "__weight__"

df = df.dropna(subset=[DIST_COL, STEPS_COL, PARK_CLASS_NAME_COL]).copy()
df = df[(df[DIST_COL] >= 0) & (df[STEPS_COL] >= 0)].copy()

df["visit_weight"] = df[WEIGHT_COL].astype(float)
df["benefit_jpy"] = df[STEPS_COL].astype(float) * STEP_VALUE_JPY * df["visit_weight"]

df["dist_bin"] = pd.cut(
    df[DIST_COL],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    include_lowest=True,
    right=True,
)

existing_types = [x for x in PARK_TYPE_ORDER if x in set(df[PARK_CLASS_NAME_COL].astype(str))]
if len(existing_types) == 0:
    existing_types = sorted(df[PARK_CLASS_NAME_COL].dropna().astype(str).unique().tolist())

df[PARK_CLASS_NAME_COL] = pd.Categorical(df[PARK_CLASS_NAME_COL], categories=existing_types, ordered=True)
df = df.sort_values(PARK_CLASS_NAME_COL).copy()

print(f"Rows after cleaning: {len(df):,}")
print(f"Park types used: {existing_types}")

distance_rows = []
for ptype, g in df.groupby(PARK_CLASS_NAME_COL, observed=True):
    dist = g[DIST_COL].to_numpy(float)
    if len(dist) == 0:
        continue
    q25, q50, q75 = np.percentile(dist, [25, 50, 75])
    p80, p90 = np.percentile(dist, [80, 90])

    distance_rows.append({
        "park_type": ptype,
        "n_visits": len(g),
        "median_distance_m": q50,
        "p25_distance_m": q25,
        "p75_distance_m": q75,
        "iqr_distance_m": q75 - q25,
        "p80_distance_m": p80,
        "p90_distance_m": p90,
        "share_within_250m_pct": 100 * (dist <= 250).mean(),
        "share_within_500m_pct": 100 * (dist <= 500).mean(),
        "share_within_1km_pct": 100 * (dist <= 1000).mean(),
        "share_within_3km_pct": 100 * (dist <= 3000).mean(),
        "share_within_6km_pct": 100 * (dist <= 6000).mean(),
        "share_beyond_6km_visits_pct": 100 * (dist > 6000).mean(),
    })

distance_summary = pd.DataFrame(distance_rows)

# Visit shares use visit weights; benefit shares use monetized step totals.
visit_bin = (
    df.groupby([PARK_CLASS_NAME_COL, "dist_bin"], observed=True)["visit_weight"]
      .sum()
      .reset_index(name="visit_weight_sum")
)

visit_totals = (
    visit_bin.groupby(PARK_CLASS_NAME_COL, observed=True)["visit_weight_sum"]
    .sum()
    .reset_index(name="visit_total")
)

visit_bin = visit_bin.merge(visit_totals, on=PARK_CLASS_NAME_COL, how="left")
visit_bin["visit_share"] = visit_bin["visit_weight_sum"] / visit_bin["visit_total"]

visit_share_wide = (
    visit_bin.pivot(index=PARK_CLASS_NAME_COL, columns="dist_bin", values="visit_share")
    .reindex(index=existing_types, columns=BIN_LABELS)
    .fillna(0.0)
)

benefit_bin = (
    df.groupby([PARK_CLASS_NAME_COL, "dist_bin"], observed=True)["benefit_jpy"]
      .sum()
      .reset_index(name="benefit_sum")
)

benefit_totals = (
    benefit_bin.groupby(PARK_CLASS_NAME_COL, observed=True)["benefit_sum"]
    .sum()
    .reset_index(name="benefit_total")
)

benefit_bin = benefit_bin.merge(benefit_totals, on=PARK_CLASS_NAME_COL, how="left")
benefit_bin["benefit_share"] = benefit_bin["benefit_sum"] / benefit_bin["benefit_total"]

benefit_share_wide = (
    benefit_bin.pivot(index=PARK_CLASS_NAME_COL, columns="dist_bin", values="benefit_share")
    .reindex(index=existing_types, columns=BIN_LABELS)
    .fillna(0.0)
)

share_summary = pd.DataFrame({"park_type": existing_types})

for lab in BIN_LABELS:
    share_summary[f"visit_share_{lab}"] = visit_share_wide[lab].values * 100
for lab in BIN_LABELS:
    share_summary[f"benefit_share_{lab}"] = benefit_share_wide[lab].values * 100

share_summary["share_beyond_6km_visits_pct"] = visit_share_wide[">6 km"].values * 100
share_summary["share_beyond_6km_benefits_pct"] = benefit_share_wide[">6 km"].values * 100

tail_df = df[df[DIST_COL] > 6000].copy()

tail_rows = []
if len(tail_df) > 0:
    for ptype, g in tail_df.groupby(PARK_CLASS_NAME_COL, observed=True):
        park_visit = (
            g.groupby("osm_id", dropna=False)["visit_weight"]
            .sum()
            .sort_values(ascending=False)
        )
        park_benefit = (
            g.groupby("osm_id", dropna=False)["benefit_jpy"]
            .sum()
            .sort_values(ascending=False)
        )

        total_visit = park_visit.sum()
        total_benefit = park_benefit.sum()

        def top_share(series, n):
            if len(series) == 0 or series.sum() == 0:
                return np.nan
            return 100 * series.head(n).sum() / series.sum()

        tail_rows.append({
            "park_type": ptype,
            "n_tail_visits": len(g),
            "top1_tail_visit_share_pct": top_share(park_visit, 1),
            "top5_tail_visit_share_pct": top_share(park_visit, 5),
            "top10_tail_visit_share_pct": top_share(park_visit, 10),
            "top1_tail_benefit_share_pct": top_share(park_benefit, 1),
            "top5_tail_benefit_share_pct": top_share(park_benefit, 5),
            "top10_tail_benefit_share_pct": top_share(park_benefit, 10),
        })

tail_summary = pd.DataFrame(tail_rows)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

n_types = len(existing_types)
base_positions = np.arange(n_types)[::-1]
y_visit = base_positions + 0.18
y_benefit = base_positions - 0.18

bar_h = 0.28

left = np.zeros(n_types)
for i, lab in enumerate(BIN_LABELS):
    vals = visit_share_wide[lab].values
    ax.barh(
        y_visit,
        vals,
        left=left,
        height=bar_h,
        color=VISIT_COLORS[i],
        edgecolor="white",
        linewidth=0.8,
        label=lab if i == 0 else None
    )

    for j, v in enumerate(vals):
        if v >= LABEL_MIN_SHARE:
            ax.text(
                left[j] + v / 2,
                y_visit[j],
                f"{v*100:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )
    left += vals

left = np.zeros(n_types)
for i, lab in enumerate(BIN_LABELS):
    vals = benefit_share_wide[lab].values
    ax.barh(
        y_benefit,
        vals,
        left=left,
        height=bar_h,
        color=BENEFIT_COLORS[i],
        edgecolor="white",
        linewidth=0.8
    )
    for j, v in enumerate(vals):
        if v >= LABEL_MIN_SHARE:
            ax.text(
                left[j] + v / 2,
                y_benefit[j],
                f"{v*100:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="black"
            )
    left += vals

ax.set_yticks(base_positions)
ax.set_yticklabels(existing_types, fontsize=11)

for i in range(n_types):
    ax.text(-0.02, y_visit[i], "Visit", ha="right", va="center", fontsize=9, transform=ax.get_yaxis_transform())
    ax.text(-0.02, y_benefit[i], "Benefit", ha="right", va="center", fontsize=9, transform=ax.get_yaxis_transform())

ax.set_xlim(0, 1)
ax.set_xlabel("Share within park type", fontsize=11)

xticks = np.linspace(0, 1, 6)
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x*100)}%" for x in xticks])

ax.set_title("Home-to-park network-distance catchment composition by park type", fontsize=13)

ax.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.35)

from matplotlib.patches import Patch

legend_bins_visit = [
    Patch(facecolor=VISIT_COLORS[i], edgecolor="white", label=BIN_LABELS[i])
    for i in range(len(BIN_LABELS))
]
legend_bins_benefit = [
    Patch(facecolor=BENEFIT_COLORS[i], edgecolor="white", label=BIN_LABELS[i])
    for i in range(len(BIN_LABELS))
]

leg1 = ax.legend(
    handles=legend_bins_visit,
    title="Visit share bins",
    bbox_to_anchor=(1.02, 1.00),
    loc="upper left",
    frameon=True
)
ax.add_artist(leg1)

leg2 = ax.legend(
    handles=legend_bins_benefit,
    title="Benefit share bins",
    bbox_to_anchor=(1.02, 0.43),
    loc="upper left",
    frameon=True
)

plt.tight_layout()

plot_path = OUTPUT_DIR / "catchment_composition_stacked_bar.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved plot: {plot_path}")

distance_summary_path = OUTPUT_DIR / "catchment_distance_summary.csv"
share_summary_path = OUTPUT_DIR / "catchment_share_summary.csv"
tail_summary_path = OUTPUT_DIR / "catchment_tail_concentration_summary.csv"

distance_summary.to_csv(distance_summary_path, index=False, encoding="utf-8-sig")
share_summary.to_csv(share_summary_path, index=False, encoding="utf-8-sig")
tail_summary.to_csv(tail_summary_path, index=False, encoding="utf-8-sig")

print(f"Saved: {distance_summary_path}")
print(f"Saved: {share_summary_path}")
print(f"Saved: {tail_summary_path}")

print("\n===== Distance summary =====")
display(distance_summary)

print("\n===== Share summary =====")
display(share_summary)

print("\n===== Tail concentration summary (>6 km) =====")
display(tail_summary)
